"""
NOTEBOOK 1: Data Preparation
Run this ONCE before any other notebook.
 
Inputs:
    - capitol_trades_cleaned.csv
    - ticker_data.csv
    - sp500_cleaned.csv
 
Outputs:
    - capitol_trades_clean.csv      (cleaned trade ledger)
    - all_stocks_clean.csv          (cleaned price data, stocks + SPY combined)
 
Usage:
    - Run cell by cell 
    - Upload all 3 input CSVs
"""

# CELL 1 — Imports & File Paths

In [3]:
import pandas as pd
import numpy as np
import os

TRADES_PATH  = "data/processed/capitol_trades_cleaned.csv"
STOCKS_PATH  = "data/processed/ticker_data.csv"
SP500_PATH   = "data/processed/sp500_cleaned.csv"
 
OUTPUT_TRADES = "data/processed/cleaned/capitol_trades_clean.csv"
OUTPUT_STOCKS = "data/processed/cleaned/all_stocks_clean.csv"
 
print("All imports loaded successfully")
print(f"Looking for trades file : {TRADES_PATH}")
print(f"Looking for stocks file : {STOCKS_PATH}")
print(f"Looking for SP500 file  : {SP500_PATH}")

All imports loaded successfully
Looking for trades file : data/processed/capitol_trades_cleaned.csv
Looking for stocks file : data/processed/ticker_data.csv
Looking for SP500 file  : data/processed/sp500_cleaned.csv


# CELL 2 — Load Raw Files & Preview

In [4]:
trades_raw = pd.read_csv(TRADES_PATH)
stocks_raw = pd.read_csv(STOCKS_PATH)
sp500_raw  = pd.read_csv(SP500_PATH)
 
print("=" * 55)
print("RAW FILE SHAPES")
print(f"  Capitol Trades : {trades_raw.shape[0]:,} rows x {trades_raw.shape[1]} cols")
print(f"  Ticker Data    : {stocks_raw.shape[0]:,} rows x {stocks_raw.shape[1]} cols")
print(f"  SP500 Data     : {sp500_raw.shape[0]:,} rows x {sp500_raw.shape[1]} cols")
print("=" * 55)
 
print("\nCapitol Trades columns  :", list(trades_raw.columns))
print("Ticker Data columns     :", list(stocks_raw.columns))
print("SP500 columns           :", list(sp500_raw.columns))
 
print("\n── Capitol Trades sample ──")
print(trades_raw.head(3).to_string())
 
print("\n── Ticker Data sample ──")
print(stocks_raw.head(3).to_string())
 
print("\n── SP500 sample ──")
print(sp500_raw.head(3).to_string())
 

RAW FILE SHAPES
  Capitol Trades : 27,027 rows x 13 cols
  Ticker Data    : 1,253,231 rows x 9 cols
  SP500 Data     : 813 rows x 6 cols

Capitol Trades columns  : ['politician', 'party', 'chamber', 'state', 'company', 'ticker', 'published', 'trade_date', 'filed_after', 'owner', 'trade_type', 'size_bracket', 'price']
Ticker Data columns     : ['Ticker', 'Date', 'Close', 'Dividends', 'High', 'Low', 'Open', 'Stock Splits', 'Volume']
SP500 columns           : ['Date', 'Close', 'High', 'Low', 'Open', 'Volume']

── Capitol Trades sample ──
     politician       party chamber state             company ticker  published trade_date filed_after   owner trade_type size_bracket    price
0   Greg Steube  Republican   House    FL            IONQ INC   IONQ  15-Apr-26  18-Mar-26     days 27  Spouse        BUY       1K-15K   $32.38
1  John Boozman  Republican  Senate    AR   Johnson & Johnson    JNJ  14-Apr-26  19-Mar-26     days 26   Joint       SELL       1K-15K  $237.60
2  John Boozman  Republican

# CELL 3 — Clean Capitol Trades

In [ ]:
trades = trades_raw.copy()
raw_count = len(trades)
print(f"Starting with {raw_count:,} raw trade rows\n")
 
def parse_date_col(series, col_name):
    """Try multiple date formats, report what worked."""
    for fmt in ['%d-%b-%y', '%d/%m/%Y', '%Y-%m-%d', '%m/%d/%Y']:
        try:
            parsed = pd.to_datetime(series, format=fmt)
            print(f"  '{col_name}' parsed with format: {fmt}")
            return parsed
        except Exception:
            continue
    print(f"  '{col_name}' using pandas auto-detect")
    try:
        return pd.to_datetime(series, errors='coerce', format='mixed')
    except TypeError:
        return pd.to_datetime(series, errors='coerce')
 
trades['trade_date']       = parse_date_col(trades['trade_date'], 'trade_date')
trades['disclosure_date']  = parse_date_col(trades['published'],  'published')

before = len(trades)
trades = trades.dropna(subset=['trade_date', 'disclosure_date'])
print(f"\n  Dropped {before - len(trades):,} rows with unparseable dates")
 
# ── Step 2: Extract filed_after_days as integer ──────────────
# Raw format is "days 27" — extract the number
trades['filed_after_days'] = (
    trades['filed_after']
    .astype(str)
    .str.extract(r'(\d+)')[0]
    .astype(float)
    .astype('Int64')   # nullable integer (handles NaN)
)

trades['filed_after_calc'] = (
    trades['disclosure_date'] - trades['trade_date']
).dt.days
 

trades['filed_after_days'] = trades['filed_after_calc']
print(f"\n  filed_after_days range: "
      f"{trades['filed_after_days'].min()} – {trades['filed_after_days'].max()} days")
 
trades['trade_type'] = trades['trade_type'].str.strip().str.upper()
print(f"\n  trade_type values found: {trades['trade_type'].unique()}")
 
trades['trade_type_encoded'] = trades['trade_type'].map({'BUY': 1, 'SELL': 0})
unmapped = trades['trade_type_encoded'].isna().sum()
if unmapped > 0:
    print(f"  WARNING: {unmapped} rows have unrecognised trade_type — will be dropped")
    trades = trades.dropna(subset=['trade_type_encoded'])

SIZE_MAP = {
    '1K-15K':     1,
    '15K-50K':    2,
    '50K-100K':   3,
    '100K-250K':  4,
    '250K-500K':  5,
    '500K-1M':    6,
    '1M-5M':      7,
    '5M-25M':     8,
}
 
trades['size_bracket'] = (
    trades['size_bracket']
    .astype(str)
    .str.strip()
    .str.replace('\u2013', '-')   # en-dash - hyphen
    .str.replace('\u2014', '-')   # em-dash - hyphen
    .str.replace(' ', '')
)
 
print(f"\n  size_bracket values found: {sorted(trades['size_bracket'].unique())}")
 
trades['size_bracket_ordinal'] = trades['size_bracket'].map(SIZE_MAP)
unmapped_size = trades['size_bracket_ordinal'].isna().sum()
if unmapped_size > 0:
    print(f"  WARNING: {unmapped_size} rows have unrecognised size_bracket:")
    print(f"  {trades[trades['size_bracket_ordinal'].isna()]['size_bracket'].unique()}")
 
if trades['price'].dtype == object:
    trades['price'] = (
        trades['price']
        .astype(str)
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.strip()
    )
    trades['price'] = pd.to_numeric(trades['price'], errors='coerce')
    print(f"\n  price column cleaned — {trades['price'].isna().sum()} NAs remain")
else:
    print(f"\n  price column already numeric — no cleaning needed")
 
trades['ticker'] = trades['ticker'].astype(str).str.strip().str.upper()
 
print("\n── Trades after cleaning ──")
print(trades[['politician', 'ticker', 'trade_date',
              'disclosure_date', 'filed_after_days',
              'trade_type_encoded', 'size_bracket_ordinal',
              'price']].head(5).to_string())
 


Starting with 27,027 raw trade rows

  'trade_date' using pandas auto-detect
  'published' using pandas auto-detect

  Dropped 0 rows with unparseable dates

  filed_after_days range: 0 – 867 days

  trade_type values found: <StringArray>
['BUY', 'SELL', 'EXCHANGE', 'RECEIVE']
Length: 4, dtype: str

  size_bracket values found: ['100K-250K', '15K-50K', '1K-15K', '1M-5M', '250K-500K', '500K-1M', '50K-100K', '5M-25M', '<1K']
  <StringArray>
['<1K']
Length: 1, dtype: str

  price column already numeric — no cleaning needed

── Trades after cleaning ──
     politician ticker trade_date disclosure_date  filed_after_days  trade_type_encoded  size_bracket_ordinal    price
0   Greg Steube   IONQ 2026-03-18      2026-04-15                28                 1.0                   1.0   $32.38
1  John Boozman    JNJ 2026-03-19      2026-04-14                26                 0.0                   1.0  $237.60
2  John Boozman   NVDA 2026-03-19      2026-04-14                26                 1.0 

# CELL 4 — Filter Pipeline (Capitol Trades)

In [7]:
print("=" * 55)
print("FILTER PIPELINE — Capitol Trades")
print("=" * 55)
 
step_counts = {'Raw': len(trades)}
 
# Filter 1: Drop unmapped size brackets
trades = trades.dropna(subset=['size_bracket_ordinal'])
step_counts['Valid size bracket'] = len(trades)
 
# Filter 2: Drop sub-$1K trades (anything below ordinal 1)
trades = trades[trades['size_bracket_ordinal'] >= 1]
step_counts['Remove sub-$1K'] = len(trades)
 
# Filter 3: Drop blind spots shorter than 5 trading days
trades = trades[trades['filed_after_days'] >= 5]
step_counts['Blind spot >= 5 days'] = len(trades)
 
# Filter 4: Drop rows with missing ticker or dates
trades = trades.dropna(subset=['ticker', 'trade_date', 'disclosure_date'])
step_counts['Valid ticker & dates'] = len(trades)
 
# Filter 5: Drop rows where ticker is empty string or 'NAN'
trades = trades[trades['ticker'].str.len() > 0]
trades = trades[trades['ticker'] != 'NAN']
step_counts['Valid ticker string'] = len(trades)
 
# Print funnel
print(f"\n{'Step':<30} {'Rows':>8} {'Dropped':>10}")
print("-" * 50)
prev = None
for step, count in step_counts.items():
    dropped = f"-{prev - count:,}" if prev is not None else ""
    print(f"{step:<30} {count:>8,} {dropped:>10}")
    prev = count
 
print(f"\nFinal clean trades: {len(trades):,} rows")
print(f"Unique tickers    : {trades['ticker'].nunique():,}")
print(f"Unique politicians: {trades['politician'].nunique():,}")

FILTER PIPELINE — Capitol Trades

Step                               Rows    Dropped
--------------------------------------------------
Raw                              26,923           
Valid size bracket               26,900        -23
Remove sub-$1K                   26,900         -0
Blind spot >= 5 days             26,566       -334
Valid ticker & dates             26,566         -0
Valid ticker string              26,566         -0

Final clean trades: 26,566 rows
Unique tickers    : 1,676
Unique politicians: 150


# CELL 5 — Clean Ticker Data

In [8]:
stocks = stocks_raw.copy()
 
# Standardise column names to lowercase
stocks.columns = [c.lower().strip() for c in stocks.columns]
print("Ticker data columns after normalising:", list(stocks.columns))
 
# Parse date
stocks['date'] = pd.to_datetime(stocks['date'], errors='coerce')
stocks = stocks.dropna(subset=['date'])
 
# Standardise ticker column
stocks['ticker'] = stocks['ticker'].astype(str).str.strip().str.upper()
 
# Keep only needed columns
stocks = stocks[['ticker', 'date', 'open', 'high', 'low', 'close', 'volume']]
 
# Drop rows where close price is missing
stocks = stocks.dropna(subset=['close'])
 
print(f"\nTicker data shape after basic cleaning: {stocks.shape}")
print(f"Date range: {stocks['date'].min().date()} → {stocks['date'].max().date()}")
print(f"Unique tickers: {stocks['ticker'].nunique():,}")

Ticker data columns after normalising: ['ticker', 'date', 'close', 'dividends', 'high', 'low', 'open', 'stock splits', 'volume']

Ticker data shape after basic cleaning: (1253231, 7)
Date range: 2023-01-02 → 2026-03-31
Unique tickers: 1,565


# CELL 6 — Clean SP500 Data & Merge Into Stocks

In [13]:
sp500 = sp500_raw.copy()
sp500.columns = [c.lower().strip() for c in sp500.columns]
 
# Parse date
sp500['date'] = pd.to_datetime(sp500['date'], format='%m/%d/%Y', errors='coerce')
# Fallback if above fails
if sp500['date'].isna().mean() > 0.5:
    try:
        sp500['date'] = pd.to_datetime(sp500_raw.iloc[:, 0], errors='coerce', format='mixed')
    except TypeError:
        sp500['date'] = pd.to_datetime(sp500_raw.iloc[:, 0], errors='coerce')
 
sp500 = sp500.dropna(subset=['date'])
sp500['ticker'] = 'SPY'  # label it as SPY for consistency
 
# Keep same columns as stocks
sp500 = sp500[['ticker', 'date', 'open', 'high', 'low', 'close', 'volume']]
sp500 = sp500.dropna(subset=['close'])
 
print(f"SP500 rows after cleaning: {len(sp500):,}")
print(f"SP500 date range: {sp500['date'].min().date()} → {sp500['date'].max().date()}")
 
# Verify SPY not already in ticker_data
if 'SPY' in stocks['ticker'].values:
    print("\nSPY already in ticker_data — removing duplicate before merge")
    stocks = stocks[stocks['ticker'] != 'SPY']
 
# Combine stocks + SPY into one file
all_stocks = pd.concat([stocks, sp500], ignore_index=True)
all_stocks = all_stocks.sort_values(['ticker', 'date']).reset_index(drop=True)
 
print(f"\nCombined stocks shape: {all_stocks.shape}")
print(f"SPY rows in combined: {(all_stocks['ticker'] == 'SPY').sum():,}")

SP500 rows after cleaning: 813
SP500 date range: 2023-01-02 → 2026-12-03

Combined stocks shape: (1254044, 7)
SPY rows in combined: 813


# CELL 7 — Drop Tickers With >10% Missing Close Prices

In [14]:
print("=" * 55)
print("FILTER — Drop tickers with >10% missing close prices")
print("=" * 55)
 
# Expected trading days in a 3-year window ≈ 756
# Count actual rows per ticker vs expected
total_days = all_stocks.groupby('ticker').size()
missing_days = all_stocks.groupby('ticker')['close'].apply(lambda x: x.isna().sum())
missing_pct = (missing_days / total_days)
 
bad_tickers = missing_pct[missing_pct > 0.10].index.tolist()
print(f"\nTickers with >10% missing close: {len(bad_tickers)}")
if bad_tickers[:10]:
    print(f"Examples: {bad_tickers[:10]}")
 
all_stocks = all_stocks[~all_stocks['ticker'].isin(bad_tickers)]
print(f"Stocks rows after dropping bad tickers: {len(all_stocks):,}")
print(f"Valid tickers remaining: {all_stocks['ticker'].nunique():,}")
 
if 'SPY' not in all_stocks['ticker'].values:
    raise ValueError(
        "CRITICAL: SPY was dropped from the dataset. "
        "Check sp500_cleaned.csv for data quality issues. "
        "Pipeline cannot continue without SPY."
    )
else:
    print("\n✓ SPY is present in all_stocks_clean — pipeline can continue")

FILTER — Drop tickers with >10% missing close prices

Tickers with >10% missing close: 0
Stocks rows after dropping bad tickers: 1,254,044
Valid tickers remaining: 1,566

✓ SPY is present in all_stocks_clean — pipeline can continue


# CELL 8 — Final Validation & Save

In [16]:
print("=" * 55)
print("FINAL VALIDATION")
print("=" * 55)
 
# Check that trades tickers overlap with stocks tickers
trade_tickers  = set(trades['ticker'].unique())
stocks_tickers = set(all_stocks['ticker'].unique())
 
overlap     = trade_tickers & stocks_tickers
missing     = trade_tickers - stocks_tickers
 
print(f"\nUnique tickers in trades          : {len(trade_tickers):,}")
print(f"Unique tickers in stocks          : {len(stocks_tickers):,}")
print(f"Tickers present in BOTH (overlap) : {len(overlap):,}")
print(f"Tickers in trades but NOT stocks  : {len(missing):,}")
 
if missing:
    print(f"\nMissing tickers (will be dropped at join in Notebook 3):")
    print(sorted(list(missing))[:30], "..." if len(missing) > 30 else "")

surviving_trades = trades[trades['ticker'].isin(stocks_tickers)]
print(f"\nEstimated trades surviving to Notebook 3: {len(surviving_trades):,}")
print(f"Estimated trades lost at join           : {len(trades) - len(surviving_trades):,}")
 
# Final column selection for trades output
trades_output = trades[[
    'politician', 'party', 'chamber', 'state',
    'company', 'ticker',
    'trade_date', 'disclosure_date', 'filed_after_days',
    'owner', 'trade_type', 'trade_type_encoded',
    'size_bracket', 'size_bracket_ordinal',
    'price'
]].copy()
 
# Save
trades_output.to_csv(OUTPUT_TRADES, index=False)
all_stocks.to_csv(OUTPUT_STOCKS, index=False)
 
print(f"\n{'='*55}")
print(f"SAVED OUTPUTS")
print(f"{'='*55}")
print(f"  {OUTPUT_TRADES}  →  {len(trades_output):,} rows")
print(f"  {OUTPUT_STOCKS}  →  {len(all_stocks):,} rows")
print(f"\nNotebook 1 complete. Proceed to Notebook 2 (EDA).")
 

FINAL VALIDATION

Unique tickers in trades          : 1,676
Unique tickers in stocks          : 1,566
Tickers present in BOTH (overlap) : 1,551
Tickers in trades but NOT stocks  : 125

Missing tickers (will be dropped at join in Notebook 3):
['1093:HK', '8376923Z:LN', 'ABB', 'ADMR', 'AGR', 'ALE', 'AMED', 'ANSS', 'ANZBY', 'APPL', 'AQUA', 'ARB:NZ', 'ARCH', 'ARGO', 'ATVI', 'AY', 'AYX', 'AZPN', 'BECN', 'BERY', 'BF/A', 'BGNE', 'BHLB', 'BRCM', 'BRK/B', 'BRY.1', 'CADE', 'CAJ', 'CCCS', 'CHUY'] ...

Estimated trades surviving to Notebook 3: 25,668
Estimated trades lost at join           : 898

SAVED OUTPUTS
  data/processed/cleaned/capitol_trades_clean.csv  →  26,566 rows
  data/processed/cleaned/all_stocks_clean.csv  →  1,254,044 rows

Notebook 1 complete. Proceed to Notebook 2 (EDA).
